In [1]:
# Cell 1: Install all libraries needed for phonetic normalization
!pip install eng-to-ipa -q
!pip install phonetics -q
!pip install indic-transliteration -q
!pip install pandas numpy tqdm -q

print("✅ All libraries installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 26.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.4/160.4 kB 4.5 MB/s eta 0:00:00
✅ All libraries installed


In [2]:
# Cell 2: Load preprocessed data from Notebook 1
import pandas as pd
import numpy as np
from tqdm import tqdm
tqdm.pandas()

# Load all three splits
train_df = pd.read_csv('/kaggle/input/datasets/ankiiitmishra/benglish-refined-data/train (1).csv')
val_df   = pd.read_csv('/kaggle/input/datasets/ankiiitmishra/benglish-refined-data/val.csv')
test_df  = pd.read_csv('/kaggle/input/datasets/ankiiitmishra/benglish-refined-data/test.csv')

print("✅ Data loaded successfully")
print(f"   Train : {len(train_df):,} rows")
print(f"   Val   : {len(val_df):,} rows")
print(f"   Test  : {len(test_df):,} rows")

print(f"\nColumns: {list(train_df.columns)}")
print(f"\nSample rows:")
train_df[['text', 'cmi', 'dominant_lang']].sample(3, random_state=42)

✅ Data loaded successfully
   Train : 37,095 rows
   Val   : 4,637 rows
   Test  : 4,637 rows

Columns: ['text', 'source', 'token_count', 'cmi', 'cmi_bucket', 'dominant_lang', 'rbn_count', 'bn_count', 'en_count']

Sample rows:


,text,cmi,dominant_lang
34376,কান্না থামাতে পারেনি জালেম ছেলে,0.0,BN_DOM
31209,এ কি মানুষ একে আমি পাইলে কেটে টুকরা টুকরা করতা...,0.0,BN_DOM
25913,Naim shikh ra bad dawa hok,0.0,EN_DOM


In [3]:
# Cell 3: Bengali Script → IPA Converter
# Maps Bengali Unicode characters to IPA phonetic symbols

# Bengali character to IPA mapping
# Based on standard Bengali phonology
BENGALI_TO_IPA = {
    # Vowels (independent)
    'অ': 'ɔ',  'আ': 'a',  'ই': 'i',  'ঈ': 'i',
    'উ': 'u',  'ঊ': 'u',  'এ': 'e',  'ঐ': 'oi',
    'ও': 'o',  'ঔ': 'ou', 'ঋ': 'ri',

    # Vowel diacritics (dependent)
    'া': 'a',  'ি': 'i',  'ী': 'i',  'ু': 'u',
    'ূ': 'u',  'ে': 'e',  'ৈ': 'oi', 'ো': 'o',
    'ৌ': 'ou', 'ৃ': 'ri',

    # Consonants
    'ক': 'k',  'খ': 'kʰ', 'গ': 'g',  'ঘ': 'gʰ', 'ঙ': 'ŋ',
    'চ': 'tʃ', 'ছ': 'tʃʰ','জ': 'dʒ', 'ঝ': 'dʒʰ','ঞ': 'n',
    'ট': 't',  'ঠ': 'tʰ', 'ড': 'd',  'ঢ': 'dʰ', 'ণ': 'n',
    'ত': 't',  'থ': 'tʰ', 'দ': 'd',  'ধ': 'dʰ', 'ন': 'n',
    'প': 'p',  'ফ': 'f',  'ব': 'b',  'ভ': 'bʰ', 'ম': 'm',
    'য': 'dʒ', 'র': 'r',  'ল': 'l',  'শ': 'ʃ',  'ষ': 'ʃ',
    'স': 's',  'হ': 'h',  'ড়': 'r', 'ঢ়': 'r', 'য়': 'j',
    'ৎ': 't',  'ং': 'ŋ',  'ঃ': 'h',  'ঁ': '~',

    # Hasanta (vowel suppressor)
    '্': '',

    # Common conjuncts
    'ক্ষ': 'kʃ', 'জ্ঞ': 'gɡ',
}

def bengali_to_ipa(word):
    """
    Convert a Bengali script word to IPA.
    Processes character by character using the mapping table.
    """
    if not isinstance(word, str):
        return ''

    ipa = []
    i = 0
    while i < len(word):
        # Check for two-character conjuncts first
        if i + 1 < len(word) and word[i:i+2] in BENGALI_TO_IPA:
            ipa.append(BENGALI_TO_IPA[word[i:i+2]])
            i += 2
        # Single character mapping
        elif word[i] in BENGALI_TO_IPA:
            ipa.append(BENGALI_TO_IPA[word[i]])
            i += 1
        else:
            # Keep character as-is if not in map (punctuation etc.)
            i += 1

    result = ''.join(ipa)

    # Post-processing: add implicit 'ɔ' after consonants
    # In Bengali, consonants have inherent 'ɔ' vowel
    # unless followed by another vowel or hasanta
    final = []
    vowels_ipa = set('aeiouɔ~')
    for j, char in enumerate(result):
        final.append(char)
        # If consonant not followed by vowel, add inherent vowel
        if char not in vowels_ipa and char != '':
            if j + 1 >= len(result) or result[j+1] not in vowels_ipa:
                final.append('ɔ')

    return ''.join(final)

# --- Test on real Bengali words ---
test_words = [
    ('ভালো',  'bhalo  - good'),
    ('আমি',   'ami    - I/me'),
    ('বাংলা', 'bangla - Bengali'),
    ('খাবার', 'khabar - food'),
    ('সুন্দর','sundor - beautiful'),
    ('করতে',  'korte  - to do'),
    ('মানুষ', 'manush - human'),
    ('দেশ',   'desh   - country'),
]

print("Bengali Script → IPA Conversion Tests")
print("=" * 50)
print(f"{'Word':<12} {'Meaning':<25} {'IPA'}")
print("-" * 50)
for word, meaning in test_words:
    ipa = bengali_to_ipa(word)
    print(f"{word:<12} {meaning:<25} /{ipa}/")

print("\n✅ Bengali → IPA converter ready")


Bengali Script → IPA Conversion Tests
Word         Meaning                   IPA
--------------------------------------------------
ভালো         bhalo  - good             /bɔʰalo/
আমি          ami    - I/me             /ami/
বাংলা        bangla - Bengali          /baŋɔla/
খাবার        khabar - food             /kɔʰabarɔ/
সুন্দর       sundor - beautiful        /sunɔdɔrɔ/
করতে         korte  - to do            /kɔrɔte/
মানুষ        manush - human            /manuʃɔ/
দেশ          desh   - country          /deʃɔ/

✅ Bengali → IPA converter ready


In [4]:
# Cell 4: Romanized Bengali → IPA Converter
# Maps Latin-script Bengali words to IPA
# Handles the inconsistent romanization problem
# e.g. bhalo / valo / balo all → /bʰalɔ/

ROMANIZED_BN_TO_IPA = {
    # Aspirated consonants (must check BEFORE simple ones)
    'bh': 'bʰ', 'dh': 'dʰ', 'gh': 'gʰ', 'jh': 'dʒʰ',
    'kh': 'kʰ', 'ph': 'f',  'th': 'tʰ', 'ch': 'tʃ',
    'sh': 'ʃ',  'zh': 'ʒ',

    # Simple consonants
    'b': 'b',  'c': 'tʃ', 'd': 'd',  'f': 'f',
    'g': 'g',  'h': 'h',  'j': 'dʒ', 'k': 'k',
    'l': 'l',  'm': 'm',  'n': 'n',  'p': 'p',
    'r': 'r',  's': 's',  't': 't',  'v': 'bʰ',
    'w': 'w',  'x': 'ks', 'y': 'j',  'z': 'z',
    'q': 'k',

    # Vowels
    'aa': 'a',  'ee': 'i',  'oo': 'u',  'ou': 'ou',
    'oi': 'oi', 'ai': 'ai', 'au': 'ou',
    'a': 'a',   'e': 'e',   'i': 'i',
    'o': 'o',   'u': 'u',
}

# Variant normalization — map spelling variants to canonical form
# This is the KEY contribution: proving bhalo/valo/balo are the same
VARIANT_MAP = {
    # ভ variants
    'valo' : 'bhalo', 'balo' : 'bhalo', 'bhalo': 'bhalo',
    'valo' : 'bhalo', 'vaalo': 'bhalo', 'bhaal': 'bhalo',

    # আমি variants
    'aami' : 'ami',   'aamee': 'ami',   'amee' : 'ami',

    # ভালো extended
    'valো' : 'bhalo', 'vhalo': 'bhalo',

    # করা variants
    'kora' : 'kora',  'kra'  : 'kora',  'korra': 'kora',

    # যাচ্ছি variants
    'jacchi': 'jacchi', 'jaachi': 'jacchi', 'jachi': 'jacchi',
    'jetesi': 'jacchi',

    # খাওয়া variants
    'khaoa' : 'khawa', 'khawa': 'khawa', 'khoa' : 'khawa',

    # আছি variants
    'aachi' : 'achi',  'achi' : 'achi',  'achhi': 'achi',

    # দেখা variants
    'dekha' : 'dekha', 'deka' : 'dekha', 'dekhা': 'dekha',

    # তুমি variants
    'tumi'  : 'tumi',  'tmi'  : 'tumi',  'tumি' : 'tumi',

    # কিন্তু variants
    'kintu' : 'kintu', 'kinto': 'kintu', 'kintu': 'kintu',

    # অনেক variants
    'onek'  : 'onek',  'onk'  : 'onek',  'anek' : 'onek',
    'anekk' : 'onek',

    # খুব variants
    'khub'  : 'khub',  'kub'  : 'khub',  'khob' : 'khub',
    'khobb' : 'khub',
}

def romanized_bn_to_ipa(word):
    """
    Convert Romanized Bengali word to IPA.
    Step 1: Normalize spelling variants
    Step 2: Apply grapheme-to-phoneme rules
    """
    if not isinstance(word, str):
        return ''

    word_lower = word.lower().strip()

    # Step 1: Normalize variant spellings
    if word_lower in VARIANT_MAP:
        word_lower = VARIANT_MAP[word_lower]

    # Step 2: Greedy left-to-right phoneme mapping
    # Always try 2-character sequences before 1-character
    ipa = []
    i = 0
    while i < len(word_lower):
        if i + 1 < len(word_lower) and word_lower[i:i+2] in ROMANIZED_BN_TO_IPA:
            ipa.append(ROMANIZED_BN_TO_IPA[word_lower[i:i+2]])
            i += 2
        elif word_lower[i] in ROMANIZED_BN_TO_IPA:
            ipa.append(ROMANIZED_BN_TO_IPA[word_lower[i]])
            i += 1
        else:
            i += 1

    return ''.join(ipa)

# --- Test: Prove variants map to same IPA ---
print("Romanized Bengali Variant → IPA Tests")
print("=" * 55)
print("KEY TEST: Do spelling variants produce the same IPA?")
print("-" * 55)

variant_groups = [
    ('ভালো variants',  ['bhalo', 'valo', 'balo', 'vaalo']),
    ('আমি variants',   ['ami', 'aami', 'amee', 'aamee']),
    ('আছি variants',   ['achi', 'aachi', 'achhi']),
    ('অনেক variants',  ['onek', 'onk', 'anek']),
    ('খুব variants',   ['khub', 'kub', 'khob']),
    ('কিন্তু variants',['kintu', 'kinto']),
]

all_pass = True
for group_name, variants in variant_groups:
    ipas = [romanized_bn_to_ipa(v) for v in variants]
    unique_ipas = set(ipas)
    status = '✅' if len(unique_ipas) == 1 else '⚠️'
    if len(unique_ipas) != 1:
        all_pass = False
    print(f"\n{status} {group_name}")
    for variant, ipa in zip(variants, ipas):
        print(f"   {variant:<12} → /{ipa}/")

print("\n" + "=" * 55)
if all_pass:
    print("✅ All variant groups map to identical IPA")
else:
    print("⚠️  Some variants still differ — review VARIANT_MAP")
print("✅ Romanized Bengali → IPA converter ready")

Romanized Bengali Variant → IPA Tests
KEY TEST: Do spelling variants produce the same IPA?
-------------------------------------------------------

✅ ভালো variants
   bhalo        → /bʰalo/
   valo         → /bʰalo/
   balo         → /bʰalo/
   vaalo        → /bʰalo/

✅ আমি variants
   ami          → /ami/
   aami         → /ami/
   amee         → /ami/
   aamee        → /ami/

✅ আছি variants
   achi         → /atʃi/
   aachi        → /atʃi/
   achhi        → /atʃi/

✅ অনেক variants
   onek         → /onek/
   onk          → /onek/
   anek         → /onek/

✅ খুব variants
   khub         → /kʰub/
   kub          → /kʰub/
   khob         → /kʰub/

✅ কিন্তু variants
   kintu        → /kintu/
   kinto        → /kintu/

✅ All variant groups map to identical IPA
✅ Romanized Bengali → IPA converter ready


In [5]:
# Cell 5: English → IPA + Unified Phonetic Normalizer
# Combines all 3 converters into one unified pipeline

import eng_to_ipa as e2i

def english_to_ipa(word):
    """
    Convert English word to IPA using eng_to_ipa library.
    Falls back to the word itself if not found.
    """
    if not isinstance(word, str):
        return ''
    result = e2i.convert(word.lower())
    # eng_to_ipa returns '*word' if word not found
    if result.startswith('*'):
        return word.lower()
    return result

def detect_script_final(token):
    """
    Detect script type of a single token.
    Returns: BN, RBN, EN, NUM, OTHER
    """
    bn_chars = sum(1 for c in token if '\u0980' <= c <= '\u09FF')
    if bn_chars > 0:
        return 'BN'
    if token.lower() in RBN_LEXICON:
        return 'RBN'
    if token.isdigit():
        return 'NUM'
    if token.isascii() and token.isalpha():
        return 'EN'
    return 'OTHER'

# RBN_LEXICON from Notebook 1 — redefined here for self-containment
RBN_LEXICON = set([
    'ami', 'tumi', 'apni', 'se', 'tara', 'amra', 'amader', 'tomader',
    'amar', 'tomar', 'apnar', 'tar', 'achi', 'acho', 'achen', 'hoi',
    'hoy', 'hoye', 'hobe', 'korbo', 'kori', 'koro', 'koren', 'korte',
    'jacchi', 'jabo', 'jao', 'jan', 'jai', 'asha', 'aschi', 'elo',
    'gelo', 'dekhte', 'dekhi', 'dekho', 'boli', 'bolo', 'bolte',
    'khabo', 'khai', 'khao', 'nao', 'nio', 'dio', 'ki', 'ke', 'na',
    'haa', 'ha', 'nei', 'nai', 'ache', 'chhilo', 'bhalo', 'valo',
    'kharap', 'sundor', 'boro', 'choto', 'onek', 'ektu', 'aro', 'ar',
    'ebong', 'kintu', 'tobe', 'jodi', 'tahole', 'theke', 'diye',
    'niye', 'kache', 'dure', 'age', 'pore', 'ekhon', 'ajke', 'aj',
    'kal', 'shob', 'kono', 'keu', 'sob', 'oi', 'ei', 'eta', 'ota',
    'bhai', 'vai', 'apu', 'dada', 'didi', 'mama', 'chacha', 'hoise',
    'hoiche', 'geche', 'jacche', 'ashche', 'korche', 'bolche',
    'dekhche', 'khacche', 'pacchi', 'parchi', 'lagche', 'laglo',
    'mone', 'bujhte', 'janina', 'jani', 'bujhi', 'manus', 'manush',
    'kaj', 'din', 'rat', 'shomoy', 'somoy', 'khub', 'besh', 'onk',
    'niye', 'kore', 'giye', 'eshe', 'boshe', 'theke', 'por', 'valo',
    'bhalo', 'moja', 'khaishi', 'khaise', 'dekhchi', 'sunchi',
    'paichi', 'jaitase', 'hoise', 'lagse', 'korso', 'bolso',
    'dekho', 'suno', 'khub', 'onek', 'boro', 'choto', 'sundor',
])

def unified_phonetic_normalizer(text):
    """
    CORE CONTRIBUTION: Unified Phonetic Normalizer
    
    Takes any Bengali-English code-mixed sentence and returns:
    1. Token-level IPA transcription
    2. Full sentence IPA string
    3. Script tags per token
    
    Handles:
    - Bengali Unicode script    → IPA via BENGALI_TO_IPA map
    - Romanized Bengali (Latin) → IPA via ROMANIZED_BN_TO_IPA + VARIANT_MAP
    - English (Latin)           → IPA via eng_to_ipa library
    """
    if not isinstance(text, str):
        return '', [], []

    tokens     = text.split()
    ipa_tokens = []
    tag_tokens = []

    for token in tokens:
        # Clean token — remove punctuation for IPA conversion
        clean_tok = token.strip('.,!?।\'\"').lower()
        tag       = detect_script_final(clean_tok)

        if tag == 'BN':
            ipa = bengali_to_ipa(clean_tok)
        elif tag == 'RBN':
            ipa = romanized_bn_to_ipa(clean_tok)
        elif tag == 'EN':
            ipa = english_to_ipa(clean_tok)
        else:
            ipa = clean_tok  # numbers, punctuation — keep as-is

        ipa_tokens.append(ipa)
        tag_tokens.append(tag)

    full_ipa = ' '.join(ipa_tokens)
    return full_ipa, ipa_tokens, tag_tokens


# --- Test on real code-mixed sentences ---
test_sentences = [
    "ami khub tired aj",
    "valo lagche everything",
    "আমি very happy today",
    "bhalo ache tumi?",
    "ei product ta really sundor",
    "Youtube ar volg gula boring hoia jaitase",
]

print("Unified Phonetic Normalizer — Full Pipeline Test")
print("=" * 60)

for sent in test_sentences:
    ipa, ipa_toks, tags = unified_phonetic_normalizer(sent)
    print(f"\nInput : {sent}")
    print(f"Tags  : {tags}")
    print(f"IPA   : {ipa}")

print("\n✅ Unified phonetic normalizer ready")

Unified Phonetic Normalizer — Full Pipeline Test

Input : ami khub tired aj
Tags  : ['RBN', 'RBN', 'EN', 'RBN']
IPA   : ami kʰub taɪərd adʒ

Input : valo lagche everything
Tags  : ['RBN', 'RBN', 'EN']
IPA   : bʰalo lagtʃe ˈɛvriˌθɪŋ

Input : আমি very happy today
Tags  : ['BN', 'EN', 'EN', 'EN']
IPA   : ami ˈvɛri ˈhæpi təˈdeɪ

Input : bhalo ache tumi?
Tags  : ['RBN', 'RBN', 'RBN']
IPA   : bʰalo atʃe tumi

Input : ei product ta really sundor
Tags  : ['RBN', 'EN', 'EN', 'EN', 'RBN']
IPA   : ei ˈprɑdəkt tɑ ˈrɪli sundor

Input : Youtube ar volg gula boring hoia jaitase
Tags  : ['EN', 'RBN', 'EN', 'EN', 'EN', 'EN', 'RBN']
IPA   : ˈjuˌtub ar volg* ˈgjulə ˈbɔrɪŋ hoia* dʒaitase

✅ Unified phonetic normalizer ready


In [6]:
# Cell 6: Phonetic Similarity Scorer
# Measures how phonetically similar two sentences are
# This is the signal that defines contrastive pairs

def levenshtein_distance(s1, s2):
    """
    Compute character-level Levenshtein distance between two IPA strings.
    Lower = more similar phonetically.
    """
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)

    prev_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            # Insertions, deletions, substitutions
            curr_row.append(min(
                prev_row[j + 1] + 1,   # deletion
                curr_row[j] + 1,        # insertion
                prev_row[j] + (c1 != c2)# substitution
            ))
        prev_row = curr_row

    return prev_row[-1]

def phonetic_similarity(text1, text2):
    """
    Compute phonetic similarity score between two sentences.
    Score range: 0.0 (completely different) to 1.0 (identical)

    Uses normalized Levenshtein distance on IPA strings.
    """
    ipa1, _, _ = unified_phonetic_normalizer(text1)
    ipa2, _, _ = unified_phonetic_normalizer(text2)

    if not ipa1 or not ipa2:
        return 0.0

    dist     = levenshtein_distance(ipa1, ipa2)
    max_len  = max(len(ipa1), len(ipa2))

    if max_len == 0:
        return 1.0

    similarity = 1 - (dist / max_len)
    return round(similarity, 4)

# --- Test 1: Spelling variants should score HIGH ---
print("TEST 1: Spelling Variants (should score HIGH ≈ 0.8–1.0)")
print("=" * 60)
variant_pairs = [
    ("ami bhalo achi",      "ami valo achi"),
    ("khub sundor",         "kub sundor"),
    ("onek din por",        "onk din por"),
    ("tumi kemon acho",     "tumi kemon achen"),
    ("aj khub happy",       "aj khub happy"),
]
for t1, t2 in variant_pairs:
    score = phonetic_similarity(t1, t2)
    status = '✅' if score >= 0.7 else '⚠️'
    print(f"{status} [{score:.4f}]  '{t1}'")
    print(f"          '{t2}'")

# --- Test 2: Different sentences should score LOW ---
print(f"\nTEST 2: Different Sentences (should score LOW ≈ 0.0–0.4)")
print("=" * 60)
different_pairs = [
    ("ami bhalo achi",      "product ta kharap"),
    ("khub sundor",         "very bad experience"),
    ("onek din por",        "i love this item"),
]
for t1, t2 in different_pairs:
    score = phonetic_similarity(t1, t2)
    status = '✅' if score <= 0.4 else '⚠️'
    print(f"{status} [{score:.4f}]  '{t1}'")
    print(f"          '{t2}'")

# --- Test 3: Script-invariant (Bengali script vs Romanized) ---
print(f"\nTEST 3: Script Invariance (Bengali script vs Romanized)")
print("=" * 60)
script_pairs = [
    ("আমি ভালো আছি",       "ami bhalo achi"),
    ("খুব সুন্দর",          "khub sundor"),
    ("অনেক দিন পর",         "onek din por"),
]
for t1, t2 in script_pairs:
    score = phonetic_similarity(t1, t2)
    status = '✅' if score >= 0.5 else '⚠️'
    print(f"{status} [{score:.4f}]  '{t1}'")
    print(f"          '{t2}'")

print("\n✅ Phonetic similarity scorer ready")

TEST 1: Spelling Variants (should score HIGH ≈ 0.8–1.0)
✅ [1.0000]  'ami bhalo achi'
          'ami valo achi'
✅ [0.8182]  'khub sundor'
          'kub sundor'
✅ [1.0000]  'onek din por'
          'onk din por'
✅ [0.8824]  'tumi kemon acho'
          'tumi kemon achen'
✅ [1.0000]  'aj khub happy'
          'aj khub happy'

TEST 2: Different Sentences (should score LOW ≈ 0.0–0.4)
✅ [0.1111]  'ami bhalo achi'
          'product ta kharap'
✅ [0.1429]  'khub sundor'
          'very bad experience'
✅ [0.1176]  'onek din por'
          'i love this item'

TEST 3: Script Invariance (Bengali script vs Romanized)
✅ [0.7778]  'আমি ভালো আছি'
          'ami bhalo achi'
✅ [0.6667]  'খুব সুন্দর'
          'khub sundor'
✅ [0.6667]  'অনেক দিন পর'
          'onek din por'

✅ Phonetic similarity scorer ready


In [7]:
# Cell 7: Contrastive Positive Pair Generator
# Generates (anchor, positive) pairs for contrastive training
# WITHOUT any human labels — purely phonetic signal

import random
random.seed(42)
np.random.seed(42)

def generate_phonetic_augmentation(text, augment_prob=0.3):
    """
    Generate a phonetically similar version of input text.
    Strategy: Replace RBN tokens with known phonetic variants.
    This creates a 'positive pair' for contrastive learning.
    """
    # Variant pool — maps canonical form to list of variants
    AUGMENT_VARIANTS = {
        'bhalo' : ['valo', 'balo', 'bhalo'],
        'valo'  : ['bhalo', 'balo', 'valo'],
        'ami'   : ['aami', 'ami'],
        'khub'  : ['kub', 'khub', 'khob'],
        'onek'  : ['onk', 'anek', 'onek'],
        'achi'  : ['aachi', 'achhi', 'achi'],
        'tumi'  : ['tumi', 'tmi'],
        'kintu' : ['kinto', 'kintu'],
        'sundor': ['sundor', 'shundor'],
        'korte' : ['korte', 'kore'],
        'dekho' : ['dekho', 'dekhो'],
        'jacchi': ['jachi', 'jaachi', 'jacchi'],
        'lagche': ['lagse', 'lagche'],
        'boro'  : ['boro', 'baro'],
        'choto' : ['choto', 'chhoto'],
        'aj'    : ['aj', 'aaj'],
        'kal'   : ['kal', 'kaal'],
        'din'   : ['din', 'deen'],
        'bhai'  : ['bhai', 'vai', 'bha'],
        'kharap': ['kharap', 'kharab', 'krap'],
        'moja'  : ['moja', 'mazza', 'maja'],
        'bhalo' : ['valo', 'bhalo', 'balo'],
    }

    tokens     = text.split()
    new_tokens = []

    for token in tokens:
        clean = token.lower().strip('.,!?।\'\"')

        # With probability augment_prob, replace with a variant
        if clean in AUGMENT_VARIANTS and random.random() < augment_prob:
            variant = random.choice(AUGMENT_VARIANTS[clean])
            # Preserve original casing style
            if token[0].isupper():
                variant = variant.capitalize()
            new_tokens.append(variant)
        else:
            new_tokens.append(token)

    return ' '.join(new_tokens)

def generate_dropout_augmentation(text, dropout_prob=0.1):
    """
    Secondary augmentation: random token dropout.
    Similar to SimCSE's approach — drop tokens randomly.
    Keeps at least 3 tokens.
    """
    tokens = text.split()
    if len(tokens) <= 3:
        return text

    new_tokens = [
        t for t in tokens
        if random.random() > dropout_prob
    ]

    # Ensure minimum length
    if len(new_tokens) < 3:
        new_tokens = tokens[:3]

    return ' '.join(new_tokens)

def generate_pair(text, strategy='phonetic'):
    """
    Generate a positive pair for a given anchor sentence.
    
    Strategies:
    - 'phonetic'  : phonetic variant augmentation (our method)
    - 'dropout'   : token dropout (SimCSE baseline)
    - 'combined'  : both phonetic + dropout
    """
    if strategy == 'phonetic':
        return generate_phonetic_augmentation(text)
    elif strategy == 'dropout':
        return generate_dropout_augmentation(text)
    elif strategy == 'combined':
        aug = generate_phonetic_augmentation(text)
        return generate_dropout_augmentation(aug)
    return text


# --- Test pair generation ---
test_sentences = [
    "ami bhalo achi aj",
    "khub sundor product",
    "tumi kemon acho",
    "onek din por dekha holo",
    "ei jinish ta kharap kintu dam kom",
]

print("Positive Pair Generation Tests")
print("=" * 60)

for sent in test_sentences:
    phonetic_aug = generate_pair(sent, strategy='phonetic')
    dropout_aug  = generate_pair(sent, strategy='dropout')
    combined_aug = generate_pair(sent, strategy='combined')

    sim_phonetic = phonetic_similarity(sent, phonetic_aug)
    sim_dropout  = phonetic_similarity(sent, dropout_aug)
    sim_combined = phonetic_similarity(sent, combined_aug)

    print(f"\nAnchor  : {sent}")
    print(f"Phonetic: {phonetic_aug:<45} sim={sim_phonetic:.4f}")
    print(f"Dropout : {dropout_aug:<45} sim={sim_dropout:.4f}")
    print(f"Combined: {combined_aug:<45} sim={sim_combined:.4f}")

print("\n✅ Positive pair generator ready")

Positive Pair Generation Tests

Anchor  : ami bhalo achi aj
Phonetic: ami bhalo aachi aj                            sim=0.8000
Dropout : ami bhalo aj                                  sim=0.7222
Combined: aami balo aachi aj                            sim=0.6364

Anchor  : khub sundor product
Phonetic: khub sundor product                           sim=1.0000
Dropout : khub sundor product                           sim=1.0000
Combined: khub sundor product                           sim=1.0000

Anchor  : tumi kemon acho
Phonetic: tmi kemon acho                                sim=0.8750
Dropout : tumi kemon acho                               sim=1.0000
Combined: tmi kemon acho                                sim=0.8750

Anchor  : onek din por dekha holo
Phonetic: anek din por dekha holo                       sim=0.9231
Dropout : onek din por dekha holo                       sim=1.0000
Combined: onek din por dekha                            sim=0.7600

Anchor  : ei jinish ta kharap kintu dam ko

In [8]:
# Fix Cell: Generate augmented text columns first
print("Generating augmented text columns...")

train_df['aug_phonetic'] = train_df['text'].apply(
    lambda x: generate_pair(x, strategy='phonetic')
)
train_df['aug_dropout'] = train_df['text'].apply(
    lambda x: generate_pair(x, strategy='dropout')
)
train_df['aug_combined'] = train_df['text'].apply(
    lambda x: generate_pair(x, strategy='combined')
)

print(f"✅ Augmented columns created")
print(f"   aug_phonetic : {train_df['aug_phonetic'].notna().sum():,} rows")
print(f"   aug_dropout  : {train_df['aug_dropout'].notna().sum():,} rows")
print(f"   aug_combined : {train_df['aug_combined'].notna().sum():,} rows")
print(f"\nSample:")
print(train_df[['text','aug_phonetic','aug_dropout']].iloc[0])

Generating augmented text columns...
✅ Augmented columns created
   aug_phonetic : 37,095 rows
   aug_dropout  : 37,095 rows
   aug_combined : 37,095 rows

Sample:
text            Burgerology এখন বাড়ীর পাশেই.বলছি তাদের ওয়ারী...
aug_phonetic    Burgerology এখন বাড়ীর পাশেই.বলছি তাদের ওয়ারী...
aug_dropout     Burgerology এখন পাশেই.বলছি তাদের ওয়ারী ব্রাঞ্...
Name: 0, dtype: object


In [9]:
# Cell 8 (Final): Smart pre-caching + GPU-aware pipeline
import eng_to_ipa as e2i
from functools import lru_cache
from tqdm import tqdm
import numpy as np

# --- Step 1: Pre-cache ALL unique English tokens first ---
print("Step 1: Pre-caching English IPA lookups...")

# Collect every unique token across all text columns
all_texts = (
    train_df['text'].tolist() +
    train_df['aug_phonetic'].tolist() +
    train_df['aug_dropout'].tolist() +
    train_df['aug_combined'].tolist() +
    val_df['text'].tolist() +
    test_df['text'].tolist()
)

# Get all unique tokens
all_tokens = set()
for text in all_texts:
    if isinstance(text, str):
        for token in text.split():
            all_tokens.add(token.lower().strip('.,!?।\'\"'))

print(f"  Unique tokens found: {len(all_tokens):,}")

# Pre-build English IPA cache using batch lookup
# eng_to_ipa.convert() can take a full string — exploit this
english_tokens = [t for t in all_tokens if t.isascii() and t.isalpha()]
print(f"  English tokens to cache: {len(english_tokens):,}")

# Batch convert all English tokens at once
english_ipa_cache = {}
batch_size = 1000
for i in tqdm(range(0, len(english_tokens), batch_size),
              desc="Caching English IPA"):
    batch = english_tokens[i:i+batch_size]
    batch_str = ' '.join(batch)
    result_str = e2i.convert(batch_str)
    result_tokens = result_str.split()
    for token, ipa in zip(batch, result_tokens):
        english_ipa_cache[token] = ipa

print(f"✅ English IPA cache built: {len(english_ipa_cache):,} entries")

# --- Step 2: Override english_to_ipa to use cache ---
def english_to_ipa_fast(word):
    word_lower = word.lower()
    if word_lower in english_ipa_cache:
        result = english_ipa_cache[word_lower]
        return word_lower if result.startswith('*') else result
    return word_lower

# Override the slow function
def unified_phonetic_normalizer_fast(text):
    if not isinstance(text, str):
        return '', [], []
    tokens     = text.split()
    ipa_tokens = []
    tag_tokens = []
    for token in tokens:
        clean = token.strip('.,!?।\'\"').lower()
        tag   = detect_script_final(clean)
        if tag == 'BN':
            ipa = bengali_to_ipa(clean)
        elif tag == 'RBN':
            ipa = romanized_bn_to_ipa(clean)
        elif tag == 'EN':
            ipa = english_to_ipa_fast(clean)
        else:
            ipa = clean
        ipa_tokens.append(ipa)
        tag_tokens.append(tag)
    return ' '.join(ipa_tokens), ipa_tokens, tag_tokens

# --- Step 3: Build sentence IPA cache ---
print("\nStep 2: Building sentence IPA cache...")

unique_sentences = set(all_texts)
print(f"  Unique sentences: {len(unique_sentences):,}")

ipa_cache  = {}
tags_cache = {}

for text in tqdm(unique_sentences, desc="IPA per sentence"):
    if isinstance(text, str):
        ipa, _, tags = unified_phonetic_normalizer_fast(text)
        ipa_cache[text]  = ipa
        tags_cache[text] = ' '.join(tags)

print(f"✅ Sentence IPA cache: {len(ipa_cache):,} entries")

# --- Step 4: Map to dataframes ---
print("\nStep 3: Mapping to dataframes...")

train_df['ipa']              = train_df['text'].map(ipa_cache)
train_df['tags']             = train_df['text'].map(tags_cache)
train_df['aug_phonetic_ipa'] = train_df['aug_phonetic'].map(ipa_cache)
train_df['aug_dropout_ipa']  = train_df['aug_dropout'].map(ipa_cache)
train_df['aug_combined_ipa'] = train_df['aug_combined'].map(ipa_cache)

val_df['ipa']   = val_df['text'].map(ipa_cache)
val_df['tags']  = val_df['text'].map(tags_cache)

test_df['ipa']  = test_df['text'].map(ipa_cache)
test_df['tags'] = test_df['text'].map(tags_cache)

print("✅ Mapped to all dataframes")

# --- Step 5: Similarity stats ---
print("\nStep 4: Similarity statistics (500 samples)...")
sample_idx = train_df.sample(500, random_state=42).index
results = {'phonetic': [], 'dropout': [], 'combined': []}

for idx in tqdm(sample_idx, desc="Similarities"):
    row = train_df.loc[idx]
    results['phonetic'].append(phonetic_similarity(row['text'], row['aug_phonetic']))
    results['dropout'].append( phonetic_similarity(row['text'], row['aug_dropout']))
    results['combined'].append(phonetic_similarity(row['text'], row['aug_combined']))

# --- Step 6: Save ---
print("\nStep 5: Saving...")
train_df.to_csv('train_phonetic.csv', index=False)
val_df.to_csv(  'val_phonetic.csv',   index=False)
test_df.to_csv( 'test_phonetic.csv',  index=False)

# --- Summary ---
print("\n" + "=" * 55)
print("        PHONETIC PROCESSING SUMMARY")
print("=" * 55)
print(f"  Train : {len(train_df):,} rows")
print(f"  Val   : {len(val_df):,} rows")
print(f"  Test  : {len(test_df):,} rows")
print(f"\n  Similarity Scores (500 samples):")
print(f"  {'Strategy':<12} {'Mean':>7} {'Std':>7} {'Min':>7} {'Max':>7}")
print(f"  {'-'*42}")
for name, scores in results.items():
    arr = np.array(scores)
    print(f"  {name:<12} {arr.mean():>7.4f} "
          f"{arr.std():>7.4f} "
          f"{arr.min():>7.4f} "
          f"{arr.max():>7.4f}")
print(f"\n  Sample row:")
row = train_df[['text','ipa','aug_phonetic','aug_phonetic_ipa']].iloc[0]
print(f"  Original : {row['text'][:55]}")
print(f"  IPA      : {row['ipa'][:55]}")
print(f"  Augmented: {row['aug_phonetic'][:55]}")
print(f"  Aug IPA  : {row['aug_phonetic_ipa'][:55]}")
print("=" * 55)
print("✅ Saved: train_phonetic.csv | val_phonetic.csv | test_phonetic.csv")

Step 1: Pre-caching English IPA lookups...
  Unique tokens found: 103,621
  English tokens to cache: 28,163


Caching English IPA: 100%|██████████| 29/29 [00:01<00:00, 15.65it/s]


✅ English IPA cache built: 28,163 entries

Step 2: Building sentence IPA cache...
  Unique sentences: 100,428


IPA per sentence: 100%|██████████| 100428/100428 [00:07<00:00, 12881.47it/s]


✅ Sentence IPA cache: 100,428 entries

Step 3: Mapping to dataframes...
✅ Mapped to all dataframes

Step 4: Similarity statistics (500 samples)...


Similarities: 100%|██████████| 500/500 [02:38<00:00,  3.15it/s]



Step 5: Saving...

        PHONETIC PROCESSING SUMMARY
  Train : 37,095 rows
  Val   : 4,637 rows
  Test  : 4,637 rows

  Similarity Scores (500 samples):
  Strategy        Mean     Std     Min     Max
  ------------------------------------------
  phonetic      0.9988  0.0076  0.9180  1.0000
  dropout       0.8968  0.1065  0.2821  1.0000
  combined      0.8989  0.1002  0.5000  1.0000

  Sample row:
  Original : Burgerology এখন বাড়ীর পাশেই.বলছি তাদের ওয়ারী ব্রাঞ্চে
  IPA      : burgerology* ekɔʰɔnɔ barirɔ paʃeibɔlɔtɔʃɔʰi taderɔ ojar
  Augmented: Burgerology এখন বাড়ীর পাশেই.বলছি তাদের ওয়ারী ব্রাঞ্চে
  Aug IPA  : burgerology* ekɔʰɔnɔ barirɔ paʃeibɔlɔtɔʃɔʰi taderɔ ojar
✅ Saved: train_phonetic.csv | val_phonetic.csv | test_phonetic.csv
